In [1]:
import polars as pl

DATASET_PATH = "/mnt/btrfs/datasets/cybersecurity/binclass/iot/IoT-23-binclass.csv"

# Helper function using native Polars expressions (super fast)
def int_to_ip(col_name):
    ip_address = pl.col(col_name).cast(pl.UInt32)
    return (
        ((ip_address // 16777216) & 255).cast(pl.Utf8) + "." +
        ((ip_address // 65536) & 255).cast(pl.Utf8) + "." +
        ((ip_address // 256) & 255).cast(pl.Utf8) + "." +
        (ip_address & 255).cast(pl.Utf8)
    )

lf = (
    pl.scan_csv(DATASET_PATH, separator=',', try_parse_dates=False, low_memory=False)
    .select([
        int_to_ip('Src IP').alias('Src IP'),
        int_to_ip('Dst IP').alias('Dst IP'),
        pl.col('Label')
    ])
    .unique()
    .sort(pl.col('Label'))
)

df_benign = lf.filter(pl.col('Label') == 'Normal').select(pl.col('Src IP'), pl.col('Dst IP')).collect()
df_attack = lf.filter(pl.col('Label') == 'Attack').select(pl.col('Src IP'), pl.col('Dst IP')).collect()
print("#Benign: ", df_benign.height, "#Attack: ", df_attack.height)

#Benign:  4124869 #Attack:  29439342


In [22]:
with pl.Config(tbl_rows=-1):
    top_benign_src_ips = df_benign['Src IP'].value_counts(sort=True).head(10)
    top_attack_src_ips = df_attack['Src IP'].value_counts(sort=True).head(10)
    benign_dst_ips_cnt = df_benign['Dst IP'].value_counts(sort=True).height
    attack_dst_ips_cnt = df_attack['Dst IP'].value_counts(sort=True).height
    print("Top benign Src IPs:\n", top_benign_src_ips)
    print("Top attack Src IPs:\n", top_attack_src_ips)
    print("Benign Dst IP rows: ", benign_dst_ips_cnt)
    print("Attack Dst IP rows: ", attack_dst_ips_cnt)
    print(df_benign['Dst IP'].filter(df_benign['Dst IP'].is_in(df_attack['Dst IP'])).value_counts(sort=True))
    print(df_attack['Dst IP'].filter(df_attack['Dst IP'].is_in(df_benign['Dst IP'])).value_counts(sort=True))

Top benign Src IPs:
 shape: (10, 2)
┌─────────────────┬─────────┐
│ Src IP          ┆ count   │
│ ---             ┆ ---     │
│ str             ┆ u32     │
╞═════════════════╪═════════╡
│ 192.168.1.196   ┆ 4101222 │
│ 192.168.100.103 ┆ 14867   │
│ 192.168.100.111 ┆ 5513    │
│ 192.168.1.197   ┆ 1612    │
│ 192.168.1.198   ┆ 167     │
│ 192.168.100.113 ┆ 21      │
│ 192.168.1.194   ┆ 17      │
│ 192.168.1.193   ┆ 14      │
│ 192.168.1.195   ┆ 13      │
│ 192.168.100.108 ┆ 11      │
└─────────────────┴─────────┘
Top attack Src IPs:
 shape: (10, 2)
┌─────────────────┬──────────┐
│ Src IP          ┆ count    │
│ ---             ┆ ---      │
│ str             ┆ u32      │
╞═════════════════╪══════════╡
│ 192.168.100.111 ┆ 27730515 │
│ 192.168.1.200   ┆ 1685416  │
│ 192.168.100.108 ┆ 12424    │
│ 192.168.1.198   ┆ 4909     │
│ 192.168.1.194   ┆ 2599     │
│ 192.168.2.5     ┆ 5        │
│ 192.168.1.197   ┆ 5        │
│ 192.168.100.103 ┆ 4        │
│ 192.168.1.1     ┆ 3        │
│ 192.168.100.

/tmp/ipykernel_57865/244495257.py:10: DeprecationWarning: `is_in` with a collection of the same datatype is ambiguous and deprecated.
Please use `implode` to return to previous behavior.

See https://github.com/pola-rs/polars/issues/22149 for more information.
  print(df_benign['Dst IP'].filter(df_benign['Dst IP'].is_in(df_attack['Dst IP'])).value_counts(sort=True))


shape: (35_684, 2)
┌─────────────────┬───────┐
│ Dst IP          ┆ count │
│ ---             ┆ ---   │
│ str             ┆ u32   │
╞═════════════════╪═══════╡
│ 192.168.100.111 ┆ 447   │
│ 192.168.1.194   ┆ 362   │
│ 192.168.1.197   ┆ 239   │
│ 192.168.2.5     ┆ 195   │
│ 192.168.1.196   ┆ 21    │
│ 147.231.100.5   ┆ 10    │
│ 192.168.100.108 ┆ 8     │
│ 192.168.1.1     ┆ 7     │
│ 5.1.56.123      ┆ 7     │
│ 192.168.1.198   ┆ 7     │
│ 78.108.102.237  ┆ 6     │
│ 224.0.0.251     ┆ 6     │
│ 94.124.107.190  ┆ 5     │
│ 8.8.8.8         ┆ 5     │
│ 217.30.75.147   ┆ 5     │
│ 82.113.53.40    ┆ 4     │
│ 1.1.1.1         ┆ 2     │
│ 206.118.190.156 ┆ 1     │
│ 197.149.231.146 ┆ 1     │
│ 129.116.125.95  ┆ 1     │
│ 79.137.173.90   ┆ 1     │
│ 41.147.48.98    ┆ 1     │
│ 174.118.219.195 ┆ 1     │
│ 169.197.201.172 ┆ 1     │
│ 201.186.228.236 ┆ 1     │
│ 41.23.33.24     ┆ 1     │
│ 156.171.218.120 ┆ 1     │
│ 47.81.191.80    ┆ 1     │
│ 174.126.125.212 ┆ 1     │
│ 108.160.159.154 ┆ 1     │
│

/tmp/ipykernel_57865/244495257.py:11: DeprecationWarning: `is_in` with a collection of the same datatype is ambiguous and deprecated.
Please use `implode` to return to previous behavior.

See https://github.com/pola-rs/polars/issues/22149 for more information.
  print(df_attack['Dst IP'].filter(df_attack['Dst IP'].is_in(df_benign['Dst IP'])).value_counts(sort=True))


shape: (35_684, 2)
┌─────────────────┬───────┐
│ Dst IP          ┆ count │
│ ---             ┆ ---   │
│ str             ┆ u32   │
╞═════════════════╪═══════╡
│ 192.168.100.111 ┆ 2527  │
│ 192.168.1.197   ┆ 30    │
│ 192.168.1.196   ┆ 11    │
│ 192.168.1.193   ┆ 7     │
│ 192.168.100.108 ┆ 5     │
│ 192.168.1.1     ┆ 5     │
│ 192.168.1.194   ┆ 3     │
│ 156.143.152.122 ┆ 2     │
│ 180.159.127.187 ┆ 2     │
│ 197.218.178.170 ┆ 2     │
│ 197.185.225.146 ┆ 2     │
│ 184.163.131.191 ┆ 2     │
│ 156.144.184.105 ┆ 2     │
│ 197.176.144.204 ┆ 2     │
│ 197.227.210.211 ┆ 2     │
│ 156.117.129.114 ┆ 2     │
│ 197.198.255.191 ┆ 2     │
│ 130.214.182.245 ┆ 2     │
│ 151.216.99.151  ┆ 2     │
│ 197.197.197.197 ┆ 2     │
│ 197.185.123.100 ┆ 2     │
│ 156.214.250.167 ┆ 2     │
│ 197.218.169.217 ┆ 2     │
│ 197.202.189.185 ┆ 2     │
│ 156.221.104.156 ┆ 2     │
│ 74.74.74.74     ┆ 2     │
│ 156.119.155.118 ┆ 2     │
│ 113.100.158.100 ┆ 2     │
│ 156.132.166.107 ┆ 2     │
│ 156.29.86.132   ┆ 2     │
│